# Jamii Afya Falcon production pipeline

This notebook is audit-first and resume-first. It does not launch a long training run unless `FALCON_RUN_MODE=stage` is explicitly set. Every child command is streamed line-by-line, and the trainer writes JSONL events, metrics, heartbeats, and Trainer checkpoints.


In [ ]:
import json, os, selectors, shutil, subprocess, sys, time
from pathlib import Path
WORK = Path('/kaggle/working')
REPO = WORK / 'adtc-llm-limited-hardware'
EXP = REPO / 'experiments' / 'falcon-production-v1'
LOG = WORK / 'falcon-production-bootstrap-logs'
BRANCH = 'research/edge35-adaptive-streaming'
RUN_MODE = os.environ.get('FALCON_RUN_MODE', 'audit')
STAGE = os.environ.get('FALCON_STAGE', 'stage_a_capability_preserving')
CONFIG = os.environ.get('FALCON_CONFIG', 'configs/falcon-production-v1.json')
CHECKPOINT_DATASET = os.environ.get('FALCON_CHECKPOINT_DATASET', '')
INIT_ADAPTER = os.environ.get('FALCON_INIT_ADAPTER', '')
LOG.mkdir(parents=True, exist_ok=True)
def streamed(command, cwd=REPO, name='command.log', env=None):
    path = LOG / name
    merged = os.environ.copy(); merged.update(env or {}); merged['PYTHONUNBUFFERED'] = '1'
    print('STREAM', ' '.join(map(str, command)), flush=True)
    with path.open('a', encoding='utf-8', buffering=1) as handle:
        proc = subprocess.Popen(command, cwd=cwd, env=merged, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        assert proc.stdout is not None
        selector = selectors.DefaultSelector(); selector.register(proc.stdout, selectors.EVENT_READ)
        started = time.monotonic()
        try:
            while True:
                ready = selector.select(timeout=30)
                if ready:
                    line = proc.stdout.readline()
                    if line:
                        stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
                        rendered = f'[{stamp}] {line}'
                        print(rendered, end='', flush=True); handle.write(rendered); handle.flush()
                    elif proc.poll() is not None:
                        break
                else:
                    stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
                    heartbeat = f'[{stamp}] HEARTBEAT child={name} pid={proc.pid} elapsed={time.monotonic()-started:.1f}s\n'
                    print(heartbeat, end='', flush=True); handle.write(heartbeat); handle.flush()
                if proc.poll() is not None:
                    for line in proc.stdout:
                        stamp = time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())
                        rendered = f'[{stamp}] {line}'
                        print(rendered, end='', flush=True); handle.write(rendered); handle.flush()
                    break
        finally:
            selector.close()
        code = proc.wait()
        print(f'EXIT={code}', flush=True); handle.write(f'EXIT={code}\n')
    if code: raise RuntimeError(f'command failed: {command}')
if not (REPO / '.git').exists():
    if REPO.exists(): shutil.rmtree(REPO)
    streamed(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/qeinstein/adtc-llm-limited-hardware.git',str(REPO)], cwd=WORK, name='clone.log')
else:
    # Kaggle may reuse a worker checkout; never trust its previous branch/SHA.
    streamed(['git','-C',str(REPO),'fetch','origin',BRANCH], cwd=WORK, name='git-refresh.log')
    streamed(['git','-C',str(REPO),'checkout','-B',BRANCH,'origin/'+BRANCH], cwd=WORK, name='git-refresh.log')
print('repo sha:', subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'], text=True, capture_output=True, check=True).stdout.strip(), flush=True)
if not (REPO / 'requirements-falcon-production.txt').exists(): raise RuntimeError('clean checkout is missing pinned production requirements')
LOG = EXP / 'kaggle-logs'; LOG.mkdir(parents=True, exist_ok=True)
streamed([sys.executable,'-m','pip','install','-q','-r','requirements-falcon-production.txt'], name='pip.log')
gpu = subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'], text=True, capture_output=True, check=False).stdout.strip()
if 'P100' in gpu:
    # P100/sm_60 cannot use the image's current Torch or bitsandbytes path.
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','numpy<2'], name='numpy-p100.log')
    # Torch >=2.6 is required by Transformers when restoring optimizer/scheduler
    # state. cu118 still supports the P100/sm_60 path used by this worker.
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','torch==2.6.0','--index-url','https://download.pytorch.org/whl/cu118'], name='torch-p100.log')
    streamed([sys.executable,'-m','pip','uninstall','-y','torchao','torchvision','torchaudio','bitsandbytes'], name='optional-uninstall.log')
    streamed([sys.executable,'-m','pip','install','-q','--upgrade','--force-reinstall','--no-deps','transformers==4.53.3','tokenizers==0.21.4','peft==0.15.2','accelerate==1.7.0'], name='hf-stack-final.log')
else:
    streamed([sys.executable,'-m','pip','install','-q','bitsandbytes==0.50.2'], name='bitsandbytes.log')
streamed([sys.executable, '-c', 'import importlib.metadata as m, torch; print({"torch":torch.__version__,"cuda":torch.version.cuda, **{x:m.version(x) for x in ("transformers","peft","accelerate")}})'], name='environment-final.log')
print(json.dumps({'run_mode': RUN_MODE, 'stage': STAGE, 'gpu': gpu, 'checkpoint_dataset_configured': bool(CHECKPOINT_DATASET), 'repo': str(REPO)}, indent=2), flush=True)

In [ ]:
DATA_DIR = EXP / 'data'
if DATA_DIR.exists(): shutil.rmtree(DATA_DIR)
streamed([sys.executable, '-u', 'scripts/audit_falcon_data.py', '--config', CONFIG, '--tokenizer', 'tiiuae/Falcon-H1-1.5B-Deep-Instruct', '--out', str(EXP / 'raw-data-audit-exact.json')], name='raw-data-audit.log')
MCQA = REPO / 'output' / 'accuracy_sft.jsonl'
if MCQA.exists() and os.environ.get('FALCON_REUSE_BUILT_DATA') != '1': MCQA.unlink()
if not MCQA.exists():
    # Public TRAIN splits only; the cap is explicit so the first clean-worker
    # audit is bounded. Increase only after measured throughput/token-share evidence.
    mcqa_cfg = json.loads((REPO / CONFIG).read_text())['data']['mcqa']
    cap = os.environ.get('FALCON_MCQA_MAX_PER_DATASET', str(mcqa_cfg['max_per_dataset']))
    streamed([sys.executable, '-u', 'scripts/build_accuracy_sft.py', '--datasets', *mcqa_cfg['datasets'], '--max-per-dataset', cap, '--letter-permutations', str(mcqa_cfg['letter_permutations']), '--seed', str(mcqa_cfg['seed']), '--fail-on-source-error', '--out', str(MCQA)], name='mcqa-build.log')
streamed([sys.executable, '-u', 'scripts/build_falcon_dataset.py', '--config', CONFIG, '--out-dir', str(DATA_DIR)], name='dataset-build.log')
manifest = json.loads((DATA_DIR / 'data_manifest.json').read_text())
print(json.dumps({k: manifest[k] for k in ('counts','token_totals','token_shares_percent','loss_token_totals','loss_token_shares_percent','facets','missing_sources')}, indent=2), flush=True)
if RUN_MODE == 'audit':
    print('AUDIT_ONLY: no training launched. Set FALCON_RUN_MODE=resume_test or stage after persistence is configured.', flush=True)

In [ ]:
if RUN_MODE == 'persistence_verify':
    # Re-check the already-uploaded checkpoint without repeating training.
    persist_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    verify_dir = EXP / 'resume-persistence-verification'
    if verify_dir.exists(): shutil.rmtree(verify_dir)
    streamed([sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', persist_dataset, '--out-dir', str(verify_dir)], env={'FALCON_CHECKPOINT_DATASET': persist_dataset}, name='persistence-only-verify.log')
    print('PERSISTENCE_ONLY_PASS: Kaggle checkpoint download contains resumable trainer state.', flush=True)
elif RUN_MODE == 'resume_test':
    # This is intentionally tiny and ephemeral: it verifies Trainer checkpoint
    # state/resume mechanics before any expensive production stage.
    test_run = EXP / 'resume-test'
    streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(test_run), '--stage', STAGE, '--max-steps', '2', '--save-steps', '1', '--allow-ephemeral'], name='resume-test-first.log')
    streamed([sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(test_run), '--stage', STAGE, '--max-steps', '4', '--save-steps', '1', '--resume-from-checkpoint', 'latest', '--allow-ephemeral'], name='resume-test-resume.log')
    ckpt_root = test_run / STAGE / 'checkpoints'
    first = json.loads((ckpt_root / 'checkpoint-2' / 'trainer_state.json').read_text())
    resumed = json.loads((ckpt_root / 'checkpoint-4' / 'trainer_state.json').read_text())
    assert first['global_step'] == 2 and resumed['global_step'] == 4
    assert any((ckpt_root / 'checkpoint-2').glob('optimizer.*')) and any((ckpt_root / 'checkpoint-2').glob('scheduler.*')) and (ckpt_root / 'checkpoint-2' / 'rng_state.pth').exists()
    persist_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    streamed([sys.executable, '-u', 'scripts/persist_checkpoint.py', '--checkpoint', str(ckpt_root / 'checkpoint-4'), '--dataset', persist_dataset, '--message', 'falcon-production-v1 resume-test persistence proof'], env={'FALCON_CHECKPOINT_DATASET': persist_dataset}, name='resume-test-persist.log')
    verify_dir = EXP / 'resume-persistence-verification'
    if verify_dir.exists(): shutil.rmtree(verify_dir)
    streamed([sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', persist_dataset, '--out-dir', str(verify_dir)], env={'FALCON_CHECKPOINT_DATASET': persist_dataset}, name='resume-test-verify.log')
    print('RESUME_TEST_PASS: global_step 2 -> 4, optimizer/scheduler/RNG state present, and checkpoint persisted/retrieved from Kaggle Dataset.', flush=True)


In [ ]:
def choose_persisted_checkpoint(dataset, purpose, requested_step=None):
    destination = EXP / purpose
    if destination.exists(): shutil.rmtree(destination)
    streamed([sys.executable, '-u', 'scripts/verify_persisted_checkpoint.py', '--dataset', dataset, '--out-dir', str(destination)], env={'FALCON_CHECKPOINT_DATASET': dataset}, name=f'{purpose}.log')
    candidates = []
    for state_path in destination.rglob('trainer_state.json'):
        state = json.loads(state_path.read_text())
        if isinstance(state.get('global_step'), int): candidates.append((state['global_step'], state_path.parent))
    if not candidates: raise RuntimeError(f'no resumable checkpoint found in {dataset}')
    if requested_step is not None:
        exact = [item for item in candidates if item[0] == int(requested_step)]
        if not exact: raise RuntimeError(f'checkpoint step {requested_step} not found in persisted dataset')
        selected = exact[0]
    else: selected = max(candidates, key=lambda item: item[0])
    print(json.dumps({'purpose': purpose, 'checkpoint': str(selected[1]), 'global_step': selected[0]}), flush=True)
    return selected[1]
if RUN_MODE == 'export':
    # Export is deliberately a separate Kaggle mode: the base model, merged
    # HF weights, F16 GGUF, and quantized GGUF never touch this workstation.
    adapter = Path(os.environ.get('FALCON_ADAPTER_PATH', '')).resolve()
    if not adapter.is_dir(): raise RuntimeError('FALCON_ADAPTER_PATH must point to a retrieved final-adapter directory')
    export_dir = EXP / 'export' / (os.environ.get('FALCON_EXPORT_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
    streamed(['bash', 'scripts/export_falcon_gguf.sh', str(adapter), str(export_dir)], env={'BASE_MODEL': 'tiiuae/Falcon-H1-1.5B-Deep-Instruct', 'MODEL_REVISION': json.loads((REPO / CONFIG).read_text())['model']['revision']}, name='export.log')
    merged = export_dir / 'merged-hf'
    fp16_eval = export_dir / 'merged-eval'
    eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(fp16_eval), '--merged-model', str(merged), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '200')]
    for battery in os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(','):
        if battery.strip(): eval_cmd += ['--battery', battery.strip()]
    streamed(eval_cmd, name='merged-eval.log')
    deployment = export_dir / 'Falcon-H1-1.5B-Deep-Instruct-Q4_K_M.gguf'
    candidate_dir = export_dir / 'quantized-eval'
    candidate_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_candidate.py', '--model', str(deployment), '--out-dir', str(candidate_dir), '--limit', os.environ.get('FALCON_FINAL_MCQA_LIMIT', '500'), '--n-ctx', '2048']
    for battery in os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(','):
        if battery.strip(): candidate_cmd += ['--battery', battery.strip()]
    streamed(candidate_cmd, name='quantized-eval.log')
    print(json.dumps({'EXPORT_COMPLETE': True, 'export_dir': str(export_dir), 'merged_eval': str(fp16_eval), 'quantized_eval': str(candidate_dir)}, indent=2), flush=True)
elif RUN_MODE == 'pilot':
    # Bounded, persisted Stage-A experiment. This is the first meaningful
    # quality run after the infrastructure gates; it is not the final run.
    pilot_dataset = CHECKPOINT_DATASET or 'toheebogunade/jamii-afya-falcon-production-checkpoints'
    pilot_id = os.environ.get('FALCON_PILOT_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime())
    run_dir = EXP / 'pilots' / pilot_id
    pilot_steps = os.environ.get('FALCON_PILOT_STEPS', '8')
    env = {'FALCON_CHECKPOINT_DATASET': pilot_dataset}
    command = [sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(run_dir), '--stage', STAGE, '--max-steps', pilot_steps, '--save-steps', '4']
    streamed(command, env=env, name=f'pilot-{STAGE}.log')
    adapter = run_dir / STAGE / 'checkpoints' / 'final-adapter'
    if not adapter.is_dir(): raise RuntimeError(f'pilot did not produce final adapter: {adapter}')
    eval_dir = run_dir / STAGE / 'pilot-eval'
    eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(eval_dir), '--adapter', str(adapter), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '64')]
    batteries = os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(',')
    for battery in batteries:
        if battery.strip(): eval_cmd += ['--battery', battery.strip()]
    streamed(eval_cmd, name='pilot-eval.log')
    print(json.dumps({'PILOT_COMPLETE': True, 'pilot_id': pilot_id, 'steps': int(pilot_steps), 'adapter': str(adapter), 'eval': str(eval_dir)}, indent=2), flush=True)
elif RUN_MODE == 'stage':
    if not CHECKPOINT_DATASET:
        raise RuntimeError('Set FALCON_CHECKPOINT_DATASET to an existing private Kaggle dataset before a production stage.')
    streamed(['kaggle', 'datasets', 'files', '-d', CHECKPOINT_DATASET], cwd=REPO, name='checkpoint-dataset-preflight.log')
    run_dir = EXP / 'runs' / (os.environ.get('FALCON_RUN_ID') or time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
    requested_step = os.environ.get('FALCON_INIT_CHECKPOINT_STEP')
    init_adapter = Path(INIT_ADAPTER) if INIT_ADAPTER else None
    resume_arg = os.environ.get('FALCON_RESUME')
    if STAGE != 'stage_a_capability_preserving' and init_adapter is None:
        init_adapter = choose_persisted_checkpoint(CHECKPOINT_DATASET, 'stage-init-checkpoint', requested_step)
    if resume_arg and (resume_arg == 'latest' or not Path(resume_arg).exists()):
        resume_dir = choose_persisted_checkpoint(CHECKPOINT_DATASET, 'resume-checkpoint', os.environ.get('FALCON_RESUME_STEP'))
        resume_arg = str(resume_dir)
    env = {'FALCON_CHECKPOINT_DATASET': CHECKPOINT_DATASET}
    command = [sys.executable, '-u', 'scripts/train_falcon_production.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--run-dir', str(run_dir), '--stage', STAGE]
    if resume_arg:
        command += ['--resume-from-checkpoint', resume_arg]
    if init_adapter:
        command += ['--init-adapter', str(init_adapter)]
    streamed(command, env=env, name=f'{STAGE}.log')
    adapter = run_dir / STAGE / 'checkpoints' / 'final-adapter'
    if adapter.exists():
        eval_dir = run_dir / STAGE / 'stage-eval'
        eval_cmd = [sys.executable, '-u', 'scripts/evaluate_falcon_hf.py', '--config', CONFIG, '--data-dir', str(DATA_DIR), '--output-dir', str(eval_dir), '--adapter', str(adapter), '--max-dev', os.environ.get('FALCON_MAX_DEV_EVAL', '200')]
        batteries = os.environ.get('FALCON_STAGE_BATTERIES', 'docs/research/falcon_probe_heldout.json,docs/research/falcon_baseline_prompts.json,data/swahili_eval_set.json').split(',')
        for battery in batteries:
            if battery.strip(): eval_cmd += ['--battery', battery.strip()]
        streamed(eval_cmd, name=f'{STAGE}-eval.log')
else:
    print('No training stage requested.', flush=True)